# Sign & Sky AI system: extract, test and validate the 17 Wolaita signs

This notebook runs the **extraction, testing and validation modules** of the Sign & Sky AI system (paper Section 3.8, steps 5–7) on all 17 documented indigenous weather signs from Offa Woreda.

**What the AI system does here, with no manual download or calculation:**

1. **Extract**: builds a space–time query for each sign and pulls daily rainfall for 1981–2026 directly from the providers' APIs: CHIRPS v2.0, CHIRPS v3.0 and ERA5-Land through Google Earth Engine, and ERA5 through the Copernicus Climate Data Store.
2. **Test**: applies each sign's pre-registered prediction (rules R1–R4, fixed 6 Nov 2026) to its target window and computes the SNS-Score.
3. **Validate**: compares each result with climatology: zero-skill benchmark, hits against chance, Heidke skill score and an exact Poisson-binomial *p*. It then stores a data trail for every test.

**What you need:** a Google account with an Earth Engine-enabled Google Cloud project (free for research: https://code.earthengine.google.com/register). A Copernicus CDS account is optional; it adds the ERA5 sensitivity check (https://cds.climate.copernicus.eu).

**How to run:** Runtime → Run all. Fill in `EE_PROJECT` (and optionally `CDS_KEY`) in Step 3 first. At the end `results.zip` downloads; send it back.

## Step 1. Create the AI system's modules
Each cell below writes one module of the system to disk. You do not need to edit them.

In [ ]:
import os
os.makedirs('/content/sign_and_sky/tests', exist_ok=True)
%cd /content/sign_and_sky

### Knowledge base: the 17 signs and their pre-registered tests  (`specs.yaml`)

In [ ]:
%%writefile specs.yaml
# Sign & Sky re-scoring: pre-registered test specification
# Fixed on 2026-11-06, BEFORE any 2025 rainfall for Wolaita was downloaded or looked at.
# Any later change must be added as a dated amendment at the bottom, never by editing above it.
# Human-readable version with the reasoning: PREREGISTRATION.md
#
# status:
#   confirmed            direction and timing follow mechanically from the field record (rules R1-R3)
#   timing_provisional   direction is clear; the window depends on a reading the team should confirm
#   direction_needed     the record says "rainfall OR drought" (or onset/intensity/failure) without saying
#                        which one the farmers read on that date. Excluded from the primary analysis until
#                        the team records it; every branch is pre-specified below so no choice is made after
#                        seeing data.

version: 1
fixed_on: "2026-11-06"
method: sns-score-v1.1   # v1 formula; adds calendar-date windows and tercile outcomes (see sns_score.py)

seasons:   # Wolaita, confirmed by the project owner
  belg:   {start: "03-01", end: "05-31"}
  kiremt: {start: "06-01", end: "09-30"}
  bega:   {start: "10-01", end: "02-28"}   # 02-29 in leap years

primary:
  dataset: chirps          # CHIRPS v2.0 daily, 0.05 deg, gauge-blended, the standard for Ethiopia
  location: offa_point     # amended 2026-11-12: Offa Woreda, 6.75 N, 37.50 E, where the fieldwork took place (was sodo_pixel)
  reference_years: 30      # WMO climatological standard; the website default of 10 is a sensitivity run
  require_complete_target: true

locations:
  offa_point: {lat: 6.75, lon: 37.50}     # Offa Woreda, as documented in the field methods
  offa_box: {lat_min: 6.61, lat_max: 6.83, lon_min: 37.40, lon_max: 37.68}   # bounds of Offa Woreda (geoBoundaries ADM3 "Ofa")
  sodo_pixel: {lat: 6.855, lon: 37.761}
  # Approximate Wolaita Zone bounding box. Replace with the zone polygon if the team has the shapefile.
  wolaita_box: {lat_min: 6.50, lat_max: 7.20, lon_min: 37.40, lon_max: 38.20}

sensitivity:
  reference_years: [10, 30]
  datasets: [chirps, chirps_v3, era5, era5_land]
  locations: [offa_point, offa_box, sodo_pixel, wolaita_box]

null_model:
  # The "climatology forecaster": issue the same prediction on the same calendar date every year
  # 1981 .. (target year - 1) and score each year against all the other years in that span
  # (leave-one-out). Gives a base rate for H, the score expected with zero skill, and a one-sided
  # empirical p-value for the 2025 score: p = (1 + #null >= observed) / (1 + n).
  first_year: 1981

signs:
  - id: 1
    name: Ants carrying eggs uphill
    category: animal
    field_score: 9
    observed: ["2025-06-10"]
    record: "Escaping; approaching seasonal rainfall/heavy rainfall"
    rule: R1
    status: confirmed
    tests:
      - {obs: "2025-06-10", outcome: heavy_rain, window_days: 7}
    sensitivity_windows: [3, 14]

  - id: 2
    name: Bird plumage colouration
    category: animal
    field_score: 6
    observed: ["2025-03-03"]
    record: "Glossy blue = favourable rainfall; greyish = poor rainfall. Greyish seen on 3 Mar 2025 (owner)."
    rule: R2
    status: confirmed
    tests:
      - {obs: "2025-03-03", outcome: below_normal, start: "2025-03-04", end: "2025-05-31"}
    sensitivity_outcomes: [dry, drought]

  - id: 3
    name: Wind pattern changes
    category: cloud_wind
    field_score: 8
    observed: ["2025-06-05", "2025-11-03"]
    record: "Direction, strength and duration; onset, intensity, or failure of seasonal rains"
    rule: R4
    status: direction_needed
    branches:
      onset:
        - {obs: "2025-06-05", outcome: rain_onset, window_days: 21}
        - {obs: "2025-11-03", outcome: rain_onset, start: "2026-03-01", end: "2026-03-31"}
      intensity:
        - {obs: "2025-06-05", outcome: above_normal, start: "2025-06-06", end: "2025-09-30"}
        - {obs: "2025-11-03", outcome: above_normal, start: "2026-03-01", end: "2026-05-31"}
      failure:
        - {obs: "2025-06-05", outcome: below_normal, start: "2025-06-06", end: "2025-09-30"}
        - {obs: "2025-11-03", outcome: below_normal, start: "2026-03-01", end: "2026-05-31"}

  - id: 4
    name: Star pattern shifts
    category: celestial
    field_score: 7.5
    observed: ["2025-03-23"]
    record: "Appearance of specific stars; prediction of seasonal rainfall or drought"
    rule: R4
    status: direction_needed
    branches:
      rainfall:
        - {obs: "2025-03-23", outcome: not_below_normal, start: "2025-03-24", end: "2025-05-31"}
      drought:
        - {obs: "2025-03-23", outcome: drought, start: "2025-03-24", end: "2025-05-31"}

  - id: 5
    name: Coloured solar or lunar rings
    category: celestial
    field_score: 6
    observed: ["2025-06-30"]
    record: "Halos around the sun or moon; possible forthcoming rainfall"
    rule: R1
    status: confirmed
    tests:
      - {obs: "2025-06-30", outcome: rain, window_days: 7}
    sensitivity_windows: [3, 14]

  - id: 6
    name: Misty and foggy conditions
    category: atmospheric
    field_score: 8
    observed: ["2025-07-02"]
    record: "Low visibility; possible rainfall during traditional seasons"
    rule: R1
    status: confirmed
    tests:
      - {obs: "2025-07-02", outcome: rain, window_days: 7}
    sensitivity_windows: [3, 14]

  - id: 7
    name: Ants marching in line
    category: animal
    field_score: 7.5
    observed: ["2025-12-10"]
    record: "Movement in a straight line; expected normal rainfall"
    rule: R2
    status: timing_provisional   # 'normal rainfall' read as the next rainy season (Belg 2026)
    tests:
      - {obs: "2025-12-10", outcome: normal_rain, start: "2026-03-01", end: "2026-05-31"}
    sensitivity_tests:
      - {obs: "2025-12-10", outcome: normal_rain, window_days: 30}

  - id: 8
    name: Star-moon alignment
    category: celestial
    field_score: 8
    observed: ["2025-02-03"]
    record: "Relative celestial positions; seasonal rainfall or drought forecast"
    rule: R4
    status: direction_needed
    branches:
      rainfall:
        - {obs: "2025-02-03", outcome: not_below_normal, start: "2025-03-01", end: "2025-05-31"}
      drought:
        - {obs: "2025-02-03", outcome: drought, start: "2025-03-01", end: "2025-05-31"}

  - id: 9
    name: Squirrel digging behaviour
    category: animal
    field_score: 6
    observed: ["2025-11-29"]
    record: "Ground squirrels actively digging; expected normal rainy season"
    rule: R2
    status: confirmed
    tests:
      - {obs: "2025-11-29", outcome: normal_rain, start: "2026-03-01", end: "2026-05-31"}
    sensitivity_tests:
      - {obs: "2025-11-29", outcome: normal_rain, window_days: 30}

  - id: 10
    name: Dispersed ant behaviour
    category: animal
    field_score: 7
    observed: ["2025-02-19"]
    record: "Scattered searching for food; possible drought"
    rule: R2
    status: confirmed
    tests:
      - {obs: "2025-02-19", outcome: drought, start: "2025-03-01", end: "2025-05-31"}
    sensitivity_tests:
      - {obs: "2025-02-19", outcome: drought, window_days: 30}

  - id: 11
    name: Dusty slow winds
    category: cloud_wind
    field_score: 8
    observed: ["2025-07-03"]
    record: "Slow breeze raising dust upward; expected normal rainfall at usual onset"
    rule: R2
    status: timing_provisional   # seen in July, after Kiremt onset; read as the rest of Kiremt
    tests:
      - {obs: "2025-07-03", outcome: normal_rain, start: "2025-07-04", end: "2025-09-30"}
    sensitivity_tests:
      - {obs: "2025-07-03", outcome: normal_rain, window_days: 30}

  - id: 12
    name: Strong east-to-west winds
    category: cloud_wind
    field_score: 7
    observed: ["2025-03-01"]
    record: "High intensity directional wind; possible drought"
    rule: R2
    status: confirmed
    tests:
      - {obs: "2025-03-01", outcome: drought, start: "2025-03-02", end: "2025-05-31"}
    sensitivity_tests:
      - {obs: "2025-03-01", outcome: drought, window_days: 30}

  - id: 13
    name: Red sun at sunrise
    category: celestial
    field_score: 6
    observed: ["2025-06-07"]   # field table says "Jue 7"; read as June (as on the website). Team to confirm.
    record: "Reddish solar appearance; expected normal rainfall"
    rule: R2
    status: timing_provisional
    tests:
      - {obs: "2025-06-07", outcome: normal_rain, start: "2025-06-08", end: "2025-09-30"}
    sensitivity_tests:
      - {obs: "2025-06-07", outcome: normal_rain, window_days: 30}
      - {obs: "2025-07-07", outcome: normal_rain, start: "2025-07-08", end: "2025-09-30"}

  - id: 14
    name: Flowering of trees
    category: plant
    field_score: 7.5
    observed: ["2025-02"]      # month only
    record: "Tree flowering patterns; seasonal transition indicator"
    rule: R3
    status: timing_provisional   # read as: the Belg rains will start in the first month of Belg
    tests:
      - {obs: "2025-02-28", outcome: rain_onset, start: "2025-03-01", end: "2025-03-31"}
    sensitivity_tests:
      - {obs: "2025-02-28", outcome: rain_onset, start: "2025-03-01", end: "2025-04-15"}

  - id: 15
    name: Frogs croaking
    category: animal
    field_score: 8
    observed: ["2025-07-04"]
    record: "Hearing frog sounds; rain is coming in this season"
    rule: R2
    status: confirmed
    tests:
      - {obs: "2025-07-04", outcome: not_below_normal, start: "2025-07-05", end: "2025-09-30"}
    sensitivity_tests:
      - {obs: "2025-07-04", outcome: rain, window_days: 7}

  - id: 16
    name: Birds migrating from far away
    category: animal
    field_score: 8.5
    observed: ["2025-11-25"]
    record: "Bird migration; rain is likely to come within a few weeks"
    rule: R1
    status: confirmed
    tests:
      - {obs: "2025-11-25", outcome: rain, window_days: 21}
    sensitivity_windows: [14, 30]

  - id: 17
    name: Dark cloud formation and movement
    category: cloud_wind
    field_score: 8
    observed: ["2025-07-08", "2025-12-03"]
    record: "Dark, watery cloud; going to rain within a few hours or days"
    rule: R1
    status: confirmed
    tests:
      - {obs: "2025-07-08", outcome: rain, window_days: 3}
      - {obs: "2025-12-03", outcome: rain, window_days: 3}
    sensitivity_windows: [1, 7]

amendments:
  - date: "2026-11-07"
    before_data: true   # no 2025 rainfall for Wolaita had been downloaded or examined
    change: >
      Primary verification measures are now standard ones, pooled over the pre-registered tests:
      proportion correct (PC), Heidke skill score with climatological chance HSS = (PC - E)/(1 - E),
      where E is the mean climatological probability b_i of each test's event, and an exact one-sided
      Poisson-binomial test of the number of hits. Reported for (a) confirmed tests and (b) confirmed
      plus timing-provisional tests. The Brier skill score against climatology is reported as a
      supplementary measure. SNS-Score is kept, unchanged, as the per-sign score shown to farmers.
      Tests, windows, outcomes, data and b_i (leave-one-out base rate 1981 .. target year - 1) are unchanged.
    reason: >
      Reviewers expect established, citable verification measures (Heidke 1926; Wilks 2019;
      Jolliffe and Stephenson 2012); the exact test is calibrated (p <= 0.05 in 2.7% of 20,000 no-skill sets).
  - date: "2026-11-12"
    before_data: true   # no 2025 rainfall for Wolaita had been downloaded or examined
    change: >
      Primary location moved from the Wolaita Sodo pixel to Offa Woreda (6.75 N, 37.50 E), where the
      283 household interviews took place (kebeles W/Dekeya, Wareza and Yakima). Offa Woreda's bounding box
      is added as an area-mean sensitivity location; the Sodo pixel and the Wolaita Zone box are kept as
      sensitivity locations. Tests, windows, outcomes and measures are unchanged.
    reason: >
      The team's recovered field-methods record places the fieldwork in Offa Woreda, about 29 km south-west
      of Sodo; rainfall should be tested where the signs were observed.

### Test module: SNS-Score  (`sns_score.py`)

In [ ]:
%%writefile sns_score.py
"""Sign & Sky verification score in Python.

`score_sign_v1` is a line-for-line port of website/src/lib/scoring.ts (METHOD_VERSION "sns-score-v1").
tests/test_equivalence.py runs both implementations on thousands of random cases and requires identical output.

`score_window` is the v1.1 generalisation used by the paper. It keeps the v1 formula unchanged
    score = 10 x (0.5*H + 0.5*P)
and adds only two things, both written down in PREREGISTRATION.md before any data were seen:
  1. a target window given by calendar dates (for season-scale predictions, e.g. "next Belg season"),
     instead of always D+1 .. D+W;
  2. three tercile outcomes used in operational seasonal forecasting (WMO; GHACOF):
       below_normal      H = p <= 1/3,  P = 1 - p
       above_normal      H = p >= 2/3,  P = p
       not_below_normal  H = p >  1/3,  P = p
With a D+1 .. D+W window and a v1 outcome, score_window returns exactly what score_sign_v1 returns.
"""
from __future__ import annotations

import datetime as _dt
import math
from dataclasses import dataclass, asdict, field
from typing import Iterable, Mapping, Optional

METHOD_VERSION = "sns-score-v1"
METHOD_VERSION_EXT = "sns-score-v1.1"

V1_OUTCOMES = ["rain", "heavy_rain", "normal_rain", "rain_onset", "dry", "drought"]
TERCILE_OUTCOMES = ["below_normal", "above_normal", "not_below_normal"]
ALL_OUTCOMES = V1_OUTCOMES + TERCILE_OUTCOMES

THRESHOLDS = dict(
    wetDayMm=1,
    heavyDayMm=20,
    rainTotalMm=10,
    onset3DayMm=20,
    dryTotalMm=5,
    droughtPct=0.2,
    normalBand=(0.25, 0.75),
    minReferenceYears=5,
    tercileLow=1 / 3,
    tercileHigh=2 / 3,
)

DEFAULT_WINDOW = dict(rain=7, heavy_rain=7, rain_onset=21, normal_rain=30, dry=30, drought=30)


# ---------- rounding identical to JavaScript Math.round (half up, toward +inf) ----------
def js_round(x: float) -> float:
    """JavaScript Math.round: nearest integer, exact halves go up."""
    f = math.floor(x)
    return float(f + 1) if x - f >= 0.5 else float(f)


def r1(x: float) -> float:
    """Math.round(x * 10) / 10"""
    return js_round(x * 10) / 10


# ---------- dates ----------
def add_days(iso: str, n: int) -> str:
    return (_dt.date.fromisoformat(iso) + _dt.timedelta(days=n)).isoformat()


def shift_years(iso: str, years: int) -> str:
    y, m, d = map(int, iso.split("-"))
    ny = y + years
    nm_first = _dt.date(ny + (m // 12), (m % 12) + 1, 1)
    last = (nm_first - _dt.timedelta(days=1)).day
    return f"{ny:04d}-{m:02d}-{min(d, last):02d}"


def window_for(observation_date: str, window_days: int):
    return add_days(observation_date, 1), add_days(observation_date, window_days)


def trust_v1(score: float) -> str:
    """Trust levels as coded in scoring.ts trustLevel() (owner decision, 5 Nov 2026)."""
    return "very_strong" if score >= 9 else "strong" if score >= 8 else "fair" if score >= 7 else "weak"


def trust_4band(score: float) -> str:
    """Trust levels as written in the website README / paper plan."""
    return "very strong" if score >= 9 else "strong" if score >= 8 else "fair" if score >= 7 else "weak"


@dataclass
class WindowStats:
    year: int
    start: str
    end: str
    totalMm: float
    maxDayMm: float
    wetDays: int
    max3DayMm: float
    complete: bool


def stats_for(days: Mapping[str, Optional[float]], start: str, end: str, year: int) -> WindowStats:
    total = 0.0
    mx = 0.0
    wet = 0
    complete = True
    vals = []
    d = start
    while d <= end:
        v = days.get(d)
        if v is None or (isinstance(v, float) and math.isnan(v)):
            complete = False
            vals.append(0.0)
        else:
            total += v
            mx = max(mx, v)
            if v >= THRESHOLDS["wetDayMm"]:
                wet += 1
            vals.append(v)
        d = add_days(d, 1)
    max3 = 0.0
    for i in range(len(vals) - 2):
        s = vals[i] + vals[i + 1] + vals[i + 2]
        wet3 = sum(1 for v in vals[i : i + 3] if v >= THRESHOLDS["wetDayMm"])
        if wet3 >= 2:
            max3 = max(max3, s)
    return WindowStats(year, start, end, r1(total), r1(mx), wet, r1(max3), complete)


def percentile_of(x: float, ref: list) -> float:
    if not ref:
        return 0.5
    below = sum(1 for v in ref if v < x)
    equal = sum(1 for v in ref if v == x)
    return (below + 0.5 * equal) / len(ref)


def _median(xs):
    s = sorted(xs)
    m = len(s) // 2
    return s[m] if len(s) % 2 else (s[m - 1] + s[m]) / 2


def classify_actual(s: WindowStats) -> str:
    if s.maxDayMm >= THRESHOLDS["heavyDayMm"]:
        return "heavy_rain"
    if s.totalMm >= THRESHOLDS["rainTotalMm"]:
        return "rain"
    if s.totalMm >= THRESHOLDS["wetDayMm"]:
        return "light_rain"
    return "dry"


@dataclass
class ScoreResult:
    version: str
    predicted: str
    windowDays: int
    target: WindowStats
    reference: list
    referenceUsed: int
    happened: bool
    percentile: float
    oriented: float
    score: float
    trust: str
    actualOutcome: str
    explanation: str

    def to_dict(self):
        d = asdict(self)
        return d


class ScoreError(ValueError):
    pass


def _evaluate(predicted, target: WindowStats, used: list):
    """The outcome switch of scoring.ts, plus the three tercile outcomes."""
    T = THRESHOLDS
    totals = [r.totalMm for r in used]
    maxes = [r.maxDayMm for r in used]
    onsets = [r.max3DayMm for r in used]
    p_total = percentile_of(target.totalMm, totals)
    happened, percentile, oriented, why = False, p_total, p_total, ""
    if predicted == "rain":
        happened = target.totalMm >= T["rainTotalMm"]
        why = f"rain counts as happened when the window total reaches {T['rainTotalMm']} mm"
    elif predicted == "heavy_rain":
        happened = target.maxDayMm >= T["heavyDayMm"]
        percentile = oriented = percentile_of(target.maxDayMm, maxes)
        why = f"heavy rain counts as happened when one day reaches {T['heavyDayMm']} mm; the comparison uses the wettest day"
    elif predicted == "rain_onset":
        happened = target.max3DayMm >= T["onset3DayMm"]
        percentile = oriented = percentile_of(target.max3DayMm, onsets)
        why = f"onset counts as happened when 3 days bring {T['onset3DayMm']} mm with at least 2 wet days"
    elif predicted == "normal_rain":
        happened = T["normalBand"][0] <= p_total <= T["normalBand"][1]
        oriented = 1 - 2 * abs(p_total - 0.5)
        why = "normal rain counts as happened when the total falls between the 25th and 75th percentile of the reference years"
    elif predicted == "dry":
        happened = target.totalMm < max(T["dryTotalMm"], 0.5 * _median(totals))
        oriented = 1 - p_total
        why = f"dry counts as happened when the total is under {T['dryTotalMm']} mm or under half the reference median"
    elif predicted == "drought":
        happened = p_total <= T["droughtPct"] or target.totalMm < T["dryTotalMm"]
        oriented = 1 - p_total
        why = "drought counts as happened when the total is at or below the 20th percentile of the reference years"
    elif predicted == "below_normal":
        happened = p_total <= T["tercileLow"]
        oriented = 1 - p_total
        why = "below normal counts as happened when the total is in the lowest third of the reference years"
    elif predicted == "above_normal":
        happened = p_total >= T["tercileHigh"]
        oriented = p_total
        why = "above normal counts as happened when the total is in the highest third of the reference years"
    elif predicted == "not_below_normal":
        happened = p_total > T["tercileLow"]
        oriented = p_total
        why = "not below normal counts as happened when the total is above the lowest third of the reference years"
    else:
        raise ScoreError(f'"{predicted}" cannot be tested automatically.')
    return happened, percentile, oriented, why


def _score(happened, oriented):
    return js_round(10 * (0.5 * (1 if happened else 0) + 0.5 * oriented) * 10) / 10


def _fmt_oriented(x: float) -> str:
    # JavaScript Number.prototype.toFixed(2): exact binary value, ties go to the larger number
    from decimal import Decimal, ROUND_HALF_UP
    return str(Decimal(x).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


def score_window(days: Mapping[str, Optional[float]], start: str, end: str, predicted: str,
                 reference_years: int, *, require_complete_target=True, version=None) -> ScoreResult:
    """Score a prediction for the target window [start, end] against the same calendar
    window in each of the `reference_years` previous years."""
    if predicted not in ALL_OUTCOMES:
        raise ScoreError(f'"{predicted}" cannot be tested automatically.')
    window_days = (_dt.date.fromisoformat(end) - _dt.date.fromisoformat(start)).days + 1
    if not (1 <= window_days <= 366):
        raise ScoreError("The window must be between 1 and 366 days.")
    target_year = int(start[:4])
    target = stats_for(days, start, end, target_year)
    if require_complete_target and not target.complete:
        raise ScoreError("The weather data for the prediction window is incomplete, so the sign cannot be scored yet.")
    reference = [stats_for(days, shift_years(start, -y), shift_years(end, -y), target_year - y)
                 for y in range(1, reference_years + 1)]
    used = [r for r in reference if r.complete]
    if len(used) < THRESHOLDS["minReferenceYears"]:
        raise ScoreError(f"Only {len(used)} complete reference years were found; at least {THRESHOLDS['minReferenceYears']} are needed.")
    happened, percentile, oriented, why = _evaluate(predicted, target, used)
    score = _score(happened, oriented)
    lower = predicted in ("dry", "drought", "below_normal")
    explanation = (
        f"{'The predicted outcome happened' if happened else 'The predicted outcome did not happen'} in {start} to {end} "
        f"({_num(target.totalMm)} mm total, wettest day {_num(target.maxDayMm)} mm; {why}). "
        f"Compared with the same window in {len(used)} earlier years, the result ranks at {int(js_round(percentile * 100))}%"
        f"{' (lower is drier)' if lower else ''}. Score = 10 × (0.5 × {1 if happened else 0} + 0.5 × {_fmt_oriented(oriented)}) = {_num(score)}."
    )
    if version is None:
        version = METHOD_VERSION if predicted in V1_OUTCOMES else METHOD_VERSION_EXT
    return ScoreResult(version, predicted, window_days, target, reference, len(used), happened,
                       percentile, oriented, score, trust_v1(score), classify_actual(target), explanation)


def _num(x: float) -> str:
    """Format a number the way JavaScript template literals do (12.0 -> '12')."""
    if float(x).is_integer():
        return str(int(x))
    return repr(float(x))


def score_sign_v1(days: Iterable[Mapping], observation_date: str, window_days: int, predicted: str,
                  reference_years: int) -> ScoreResult:
    """Exact port of scoreSign() in scoring.ts. `days` is a list of {"date": iso, "mm": float|None}."""
    if predicted not in V1_OUTCOMES:
        raise ScoreError(f'"{predicted}" cannot be tested automatically.')
    if not (1 <= window_days <= 120):
        raise ScoreError("The window must be between 1 and 120 days.")
    m = {d["date"]: d["mm"] for d in days}
    start, end = window_for(observation_date, window_days)
    return score_window(m, start, end, predicted, reference_years, version=METHOD_VERSION)

### Validation module: standard verification measures  (`standard_metrics.py`)

In [ ]:
%%writefile standard_metrics.py
"""Standard forecast-verification measures for a set of dated categorical predictions.

Each pre-registered test i is a deterministic "yes" forecast that its event will happen in its window.
  o_i = 1 if the event happened (the H of SNS-Score), else 0
  b_i = climatological probability of the event: the fraction of earlier years (same calendar window,
        1981 .. target year - 1) in which it happened, each year judged against the others (leave-one-out)

Pooled over n tests (Wilks 2019; Jolliffe & Stephenson 2012):
  PC   = (1/n) sum o_i                                   proportion correct (hit rate)
  E    = (1/n) sum b_i                                   proportion correct expected by chance
  HSS  = (PC - E) / (1 - E)                              Heidke skill score (1 = perfect, 0 = chance)
  BS   = (1/n) sum (1 - o_i)^2                           Brier score of the forecasts (f_i = 1)
  BSc  = (1/n) sum (b_i - o_i)^2                         Brier score of the climatological forecast
  BSS  = 1 - BS / BSc                                    Brier skill score vs climatology
Significance: under no skill the number of hits C = sum o_i follows a Poisson-binomial distribution
with success probabilities b_i; one-sided exact p = Pr(C >= c_observed).
"""
from __future__ import annotations


def poisson_binomial_sf(probs, c):
    """Pr(sum of independent Bernoulli(p_i) >= c), exact by dynamic programming."""
    dist = [1.0]
    for p in probs:
        new = [0.0] * (len(dist) + 1)
        for k, v in enumerate(dist):
            new[k] += v * (1 - p)
            new[k + 1] += v * p
        dist = new
    return float(sum(dist[c:])) if c < len(dist) else 0.0


def verify(outcomes, base_rates):
    """outcomes: list of 0/1; base_rates: list of climatological probabilities b_i."""
    n = len(outcomes)
    if n == 0:
        return None
    pc = sum(outcomes) / n
    e = sum(base_rates) / n
    hss = (pc - e) / (1 - e) if e < 1 else None
    bs = sum((1 - o) ** 2 for o in outcomes) / n
    bsc = sum((b - o) ** 2 for b, o in zip(base_rates, outcomes)) / n
    bss = (1 - bs / bsc) if bsc > 0 else None
    p = poisson_binomial_sf(base_rates, int(sum(outcomes)))
    return dict(n=n, hits=int(sum(outcomes)), expected_hits=round(sum(base_rates), 2), PC=round(pc, 3),
                PC_chance=round(e, 3), HSS=None if hss is None else round(hss, 3),
                BS=round(bs, 3), BS_clim=round(bsc, 3), BSS=None if bss is None else round(bss, 3),
                p_exact=round(p, 4))

### Test and validate all signs  (`rescore.py`)

In [ ]:
%%writefile rescore.py
"""Re-score the 17 Wolaita indicators exactly as pre-registered in specs.yaml.

Reads data/<dataset>__<location>.csv (made by fetch_data.py) and writes results/:
  primary.csv        one row per pre-registered test (primary dataset, location, reference years)
  signs.csv          one row per sign: field score, re-scored score, zero-skill baseline, p-value
  branches.csv       every branch of the 'direction needed' signs (NOT part of the primary analysis)
  sensitivity.csv    every test under every dataset x location x reference-years combination,
                     plus the per-sign alternative windows / outcomes
  null.csv           the zero-skill (climatology) score for every test and hindcast year
  trail/*.json       full data trail per test: daily values of the target and reference windows,
                     thresholds, explanation (what a reviewer needs to recompute any number)
  summary.md         the headline numbers in words
  fig_scores.png, fig_windows.png, fig_sensitivity.png

Usage:  python rescore.py [--data DIR] [--out DIR]
"""
from __future__ import annotations

import argparse
import csv
import datetime as dt
import json
import math
import os
import statistics as st

import yaml

from sns_score import (THRESHOLDS, ScoreError, _evaluate, _score, add_days, score_window, shift_years,
                       stats_for, trust_4band, METHOD_VERSION_EXT)

HERE = os.path.dirname(os.path.abspath(__file__))


# ---------------------------------------------------------------- loading
def load_series(data_dir):
    out = {}
    if not os.path.isdir(data_dir):
        return out
    for fn in sorted(os.listdir(data_dir)):
        if fn.endswith(".csv") and "__" in fn:
            ds, loc = fn[:-4].split("__", 1)
            m = {}
            with open(os.path.join(data_dir, fn), encoding="utf-8") as f:
                for row in csv.DictReader(f):
                    m[row["date"]] = float(row["mm"]) if row["mm"] not in ("", None) else None
            out[(ds, loc)] = m
    return out


def window_of(test):
    if "start" in test:
        return test["start"], test["end"]
    return add_days(test["obs"], 1), add_days(test["obs"], int(test["window_days"]))


# ---------------------------------------------------------------- null model
def null_scores(days, start, end, outcome, first_year):
    """Climatology forecaster: same prediction, same calendar window, every earlier year,
    each scored against all the other earlier years (leave-one-out)."""
    ty = int(start[:4])
    stats = {}
    for y in range(first_year, ty):
        s = stats_for(days, shift_years(start, y - ty), shift_years(end, y - ty), y)
        if s.complete and int(s.start[:4]) >= first_year:
            stats[y] = s
    out = []
    for y, s in stats.items():
        ref = [r for yy, r in stats.items() if yy != y]
        if len(ref) < THRESHOLDS["minReferenceYears"]:
            continue
        h, _, o, _ = _evaluate(outcome, s, ref)
        out.append((y, h, _score(h, o)))
    return out


def p_value(observed, null):
    if not null:
        return None
    return (1 + sum(1 for _, _, sc in null if sc >= observed)) / (1 + len(null))


def fisher(ps):
    """Fisher's combined probability test; chi-square survival with 2k df in closed form."""
    ps = [p for p in ps if p is not None]
    if not ps:
        return None, 0
    x = -2 * sum(math.log(p) for p in ps)
    k = len(ps)
    term, s = 1.0, 1.0
    for i in range(1, k):
        term *= (x / 2) / i
        s += term
    return min(1.0, math.exp(-x / 2) * s), k


# ---------------------------------------------------------------- one test
def run_test(days, test, n_ref, first_year, with_null=True):
    start, end = window_of(test)
    r = score_window(days, start, end, test["outcome"], n_ref)
    null = null_scores(days, start, end, test["outcome"], first_year) if with_null else []
    base_rate = (sum(1 for _, h, _ in null if h) / len(null)) if null else None
    null_mean = st.mean(sc for _, _, sc in null) if null else None
    return r, null, base_rate, null_mean, p_value(r.score, null)


def trail(days, r, test, spec_meta):
    def window_days(a, b):
        out, d = [], a
        while d <= b:
            out.append({"date": d, "mm": days.get(d)}); d = add_days(d, 1)
        return out
    return dict(
        method=METHOD_VERSION_EXT if r.version != "sns-score-v1" else r.version,
        **spec_meta, test=test, thresholds=THRESHOLDS,
        target=dict(stats=r.target.__dict__, daily=window_days(r.target.start, r.target.end)),
        reference=[dict(stats=x.__dict__, daily=window_days(x.start, x.end)) for x in r.reference],
        happened=r.happened, percentile=r.percentile, oriented=r.oriented, score=r.score,
        explanation=r.explanation)


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default=os.path.join(HERE, "data"))
    ap.add_argument("--out", default=os.path.join(HERE, "results"))
    ap.add_argument("--label", default="", help="text added to summary/figure titles (e.g. SYNTHETIC)")
    a = ap.parse_args()

    spec = yaml.safe_load(open(os.path.join(HERE, "specs.yaml"), encoding="utf-8"))
    series = load_series(a.data)
    P = spec["primary"]
    first_year = spec["null_model"]["first_year"]
    key = (P["dataset"], P["location"])
    os.makedirs(os.path.join(a.out, "trail"), exist_ok=True)
    if key not in series:
        raise SystemExit(f"Primary series data/{key[0]}__{key[1]}.csv not found. Run fetch_data.py first. "
                         f"Found: {sorted(series)}")
    days = series[key]

    primary_rows, sign_rows, branch_rows, sens_rows, null_rows = [], [], [], [], []

    for s in spec["signs"]:
        meta = dict(sign_id=s["id"], sign=s["name"], status=s["status"], rule=s["rule"])
        # -------- primary
        if s["status"] != "direction_needed":
            scores, ps, nms, brs = [], [], [], []
            for i, t in enumerate(s["tests"]):
                try:
                    r, null, br, nm, p = run_test(days, t, P["reference_years"], first_year)
                except ScoreError as e:
                    primary_rows.append({**meta, "test": i + 1, "obs": t["obs"], "outcome": t["outcome"], "error": str(e)})
                    continue
                scores.append(r.score); ps.append(p); nms.append(nm); brs.append(br)
                primary_rows.append({**meta, "test": i + 1, "obs": t["obs"], "outcome": t["outcome"],
                                     "start": r.target.start, "end": r.target.end, "total_mm": r.target.totalMm,
                                     "max_day_mm": r.target.maxDayMm, "ref_years": r.referenceUsed,
                                     "happened": int(r.happened), "percentile": round(r.percentile, 3),
                                     "P": round(r.oriented, 3), "score": r.score, "trust": trust_4band(r.score),
                                     "base_rate": None if br is None else round(br, 3), "b": br,
                                     "null_mean": None if nm is None else round(nm, 2),
                                     "p_value": None if p is None else round(p, 3), "error": ""})
                for y, h, sc in null:
                    null_rows.append({"sign_id": s["id"], "test": i + 1, "year": y, "happened": int(h), "score": sc})
                json.dump(trail(days, r, t, {**meta, "dataset": key[0], "location": key[1]}),
                          open(os.path.join(a.out, "trail", f"sign{s['id']:02d}_test{i + 1}.json"), "w"), indent=1)
            if scores:
                sc = round(st.mean(scores), 2)
                # combine the tests of a two-observation sign by the mean score and the larger p-value (conservative)
                sign_rows.append({**meta, "category": s["category"], "field_score": s["field_score"], "rescored": sc,
                                  "trust": trust_4band(sc), "null_mean": round(st.mean(nms), 2),
                                  "skill_vs_null": round(sc - st.mean(nms), 2),
                                  "base_rate": round(st.mean(brs), 2), "p_value": round(max(ps), 3), "n_tests": len(scores)})
        # -------- branches of direction-needed signs (reported, never pooled)
        for bname, tests in (s.get("branches") or {}).items():
            for i, t in enumerate(tests):
                try:
                    r, null, br, nm, p = run_test(days, t, P["reference_years"], first_year)
                    branch_rows.append({**meta, "branch": bname, "obs": t["obs"], "outcome": t["outcome"],
                                        "start": r.target.start, "end": r.target.end, "total_mm": r.target.totalMm,
                                        "percentile": round(r.percentile, 3), "score": r.score,
                                        "null_mean": None if nm is None else round(nm, 2),
                                        "p_value": None if p is None else round(p, 3)})
                except ScoreError as e:
                    branch_rows.append({**meta, "branch": bname, "obs": t["obs"], "outcome": t["outcome"], "error": str(e)})

        # -------- sensitivity
        all_tests = list(s.get("tests") or [])
        for (ds, loc), dd in sorted(series.items()):
            for n_ref in spec["sensitivity"]["reference_years"]:
                for i, t in enumerate(all_tests):
                    sens_rows.append(_sens(meta, f"{ds}/{loc}/N{n_ref}", dd, t, n_ref, "pre-registered test", i + 1))
        for i, t in enumerate(all_tests):
            for w in s.get("sensitivity_windows", []):
                t2 = {**t, "window_days": w}; t2.pop("start", None); t2.pop("end", None)
                sens_rows.append(_sens(meta, "primary", days, t2, P["reference_years"], f"window {w} d", i + 1))
            for o in s.get("sensitivity_outcomes", []):
                sens_rows.append(_sens(meta, "primary", days, {**t, "outcome": o}, P["reference_years"], f"outcome {o}", i + 1))
        for j, t in enumerate(s.get("sensitivity_tests", [])):
            desc = f"{t['outcome']} " + (f"{t['start']}..{t['end']}" if "start" in t else f"{t['window_days']} d")
            sens_rows.append(_sens(meta, "primary", days, t, P["reference_years"], f"alt: {desc}", j + 1))

    # ---------------------------------------------------------------- write tables
    def wcsv(name, rows):
        if not rows:
            return
        cols = []
        for r in rows:
            for k in r:
                if k not in cols:
                    cols.append(k)
        with open(os.path.join(a.out, name), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)

    for name, rows in [("primary.csv", primary_rows), ("signs.csv", sign_rows), ("branches.csv", branch_rows),
                       ("sensitivity.csv", sens_rows), ("null.csv", null_rows)]:
        wcsv(name, rows)

    # ---------------------------------------------------------------- summary
    # standard verification measures, pooled over the pre-registered tests (primary analysis)
    from standard_metrics import verify
    std_rows = []
    for label, keep in [("confirmed", ("confirmed",)), ("confirmed + timing provisional", ("confirmed", "timing_provisional"))]:
        rows = [r for r in primary_rows if not r.get("error") and r["status"] in keep and r.get("b") is not None]
        v = verify([r["happened"] for r in rows], [r["b"] for r in rows])
        if v:
            std_rows.append({"set": label, **v})
    for r in primary_rows:
        r.pop("b", None)
    wcsv("standard.csv", std_rows)
    comb_p, k = fisher([r["p_value"] for r in sign_rows if r["status"] == "confirmed"])
    comb_all, k_all = fisher([r["p_value"] for r in sign_rows])
    rho = spearman_rfc(sign_rows, os.path.join(a.data, "rfc.csv"))
    write_summary(a, spec, key, sign_rows, branch_rows, sens_rows, comb_p, k, comb_all, k_all, rho, std_rows)
    try:
        import figures
        figures.make_all(a.out, sign_rows, primary_rows, null_rows, sens_rows, days, spec, a.label)
    except ImportError as e:
        print("figures skipped:", e)
    print(open(os.path.join(a.out, "summary.md"), encoding="utf-8").read())


def _sens(meta, config, days, t, n_ref, variant, i):
    try:
        start, end = window_of(t)
        r = score_window(days, start, end, t["outcome"], n_ref)
        return {**meta, "config": config, "variant": variant, "test": i, "outcome": t["outcome"],
                "start": start, "end": end, "ref_years": r.referenceUsed, "total_mm": r.target.totalMm,
                "happened": int(r.happened), "score": r.score, "error": ""}
    except ScoreError as e:
        return {**meta, "config": config, "variant": variant, "test": i, "outcome": t["outcome"], "error": str(e)}


def spearman_rfc(sign_rows, path):
    """If data/rfc.csv exists (columns: sign_id, mentions, n_farmers) test whether widely shared signs score higher."""
    if not os.path.exists(path):
        return None
    rfc = {int(r["sign_id"]): int(r["mentions"]) / int(r["n_farmers"]) for r in csv.DictReader(open(path))}
    pairs = [(rfc[r["sign_id"]], r["rescored"]) for r in sign_rows if r["sign_id"] in rfc]
    if len(pairs) < 4:
        return None

    def ranks(xs):
        order = sorted(range(len(xs)), key=lambda i: xs[i])
        rk = [0.0] * len(xs); i = 0
        while i < len(xs):
            j = i
            while j + 1 < len(xs) and xs[order[j + 1]] == xs[order[i]]:
                j += 1
            for q in range(i, j + 1):
                rk[order[q]] = (i + j) / 2 + 1
            i = j + 1
        return rk
    a, b = ranks([p[0] for p in pairs]), ranks([p[1] for p in pairs])
    ma, mb = st.mean(a), st.mean(b)
    num = sum((x - ma) * (y - mb) for x, y in zip(a, b))
    den = math.sqrt(sum((x - ma) ** 2 for x in a) * sum((y - mb) ** 2 for y in b))
    return (num / den if den else None), len(pairs)


def write_summary(a, spec, key, sign_rows, branch_rows, sens_rows, comb_p, k, comb_all, k_all, rho, std_rows=()):
    L = []
    lab = f" ({a.label})" if a.label else ""
    L.append(f"# Re-scoring results{lab}\n")
    L.append(f"Generated {dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')} from `specs.yaml` "
             f"(fixed {spec['fixed_on']}). Primary: {key[0]} at {key[1]}, {spec['primary']['reference_years']} reference years, "
             f"method {spec['method']}.\n")
    if a.label:
        L.append(f"> **{a.label}: these numbers come from made-up rainfall and mean nothing about Wolaita.** "
                 "They only show that every step of the pipeline runs.\n")
    L.append("| # | Sign | Status | Field score | Re-scored | Trust | Zero-skill score | Skill vs zero-skill | Outcome base rate | p |")
    L.append("|---|---|---|---|---|---|---|---|---|---|")
    for r in sign_rows:
        L.append(f"| {r['sign_id']} | {r['sign']} | {r['status']} | {r['field_score']} | {r['rescored']} | {r['trust']} | "
                 f"{r['null_mean']} | {r['skill_vs_null']:+} | {r['base_rate']} | {r['p_value']} |")
    if sign_rows:
        fs = [r["field_score"] for r in sign_rows]; rs = [r["rescored"] for r in sign_rows]
        L.append("")
        L.append(f"- Signs scored in the primary analysis: {len(sign_rows)} "
                 f"({sum(1 for r in sign_rows if r['status'] == 'confirmed')} confirmed, "
                 f"{sum(1 for r in sign_rows if r['status'] == 'timing_provisional')} timing provisional). "
                 f"{sum(1 for s in spec['signs'] if s['status'] == 'direction_needed')} signs await the direction the farmers read (see branches.csv).")
        L.append(f"- Mean field score {st.mean(fs):.2f}; mean re-scored {st.mean(rs):.2f}; "
                 f"mean absolute difference {st.mean(abs(x - y) for x, y in zip(fs, rs)):.2f}.")
        L.append(f"- Signs beating the zero-skill score: {sum(1 for r in sign_rows if r['skill_vs_null'] > 0)} of {len(sign_rows)}; "
                 f"with p < 0.10: {sum(1 for r in sign_rows if r['p_value'] < 0.10)}.")
        if comb_p is not None:
            L.append(f"- Fisher combined p, confirmed signs (k={k}): {comb_p:.3f}; all scored signs (k={k_all}): {comb_all:.3f}. "
                     "Signs share one place and one year, so these are optimistic.")
    if std_rows:
        L.append("\n## Standard verification (primary)\n")
        L.append("| Set | n tests | Hits | Hits expected by chance | PC | PC by chance | HSS | BSS | Exact p (hits) |")
        L.append("|---|---|---|---|---|---|---|---|---|")
        for v in std_rows:
            L.append(f"| {v['set']} | {v['n']} | {v['hits']} | {v['expected_hits']} | {v['PC']} | {v['PC_chance']} | {v['HSS']} | {v['BSS']} | {v['p_exact']} |")
        L.append("")
    if rho:
        L.append(f"- Spearman correlation between citation frequency (RFC) and re-scored score: {rho[0]:.2f} (n={rho[1]}).")
    # sensitivity: spread of each sign's score across data configurations
    by = {}
    for r in sens_rows:
        if r.get("error") == "" and r["variant"] == "pre-registered test":
            by.setdefault(r["sign_id"], []).append(r["score"])
    if by:
        spread = {s: (min(v), max(v)) for s, v in by.items()}
        L.append(f"- Across datasets x locations x reference periods, the median per-sign score range is "
                 f"{st.median(b - a_ for a_, b in spread.values()):.1f} points.")
    open(os.path.join(a.out, "summary.md"), "w", encoding="utf-8").write("\n".join(L) + "\n")


if __name__ == "__main__":
    main()

### Figures  (`figures.py`)

In [ ]:
%%writefile figures.py
"""Paper figures for the re-scoring (called by rescore.py). Static PNG + SVG, 300 dpi."""
import os
import statistics as st

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402

from sns_score import add_days, shift_years, stats_for  # noqa: E402

INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE = "#2a78d6", "#eb6834"
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
BAND_COL = {"weak": "#f3f2ee", "fair": "#ecebe6", "strong": "#e2e1db", "very strong": "#d6d5ce"}

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF})


def _save(fig, out, name):
    fig.savefig(os.path.join(out, name + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(out, name + ".svg"), bbox_inches="tight")
    plt.close(fig)


def fig_scores(out, sign_rows, label):
    rows = sorted(sign_rows, key=lambda r: r["rescored"])
    n = len(rows)
    fig, ax = plt.subplots(figsize=(7.2, 0.32 * n + 1.4))
    for lo, hi, name in [(0, 7, "weak"), (7, 8, "fair"), (8, 9, "strong"), (9, 10, "very strong")]:
        ax.axvspan(lo, hi, color=BAND_COL[name], zorder=0, lw=0)
        ax.text((lo + hi) / 2 if lo else 3.5, n - 0.35, name, ha="center", va="bottom", color=INK2, fontsize=7.5)
    y = range(n)
    h = 0.36
    ax.barh([i + h / 2 + 0.02 for i in y], [r["field_score"] for r in rows], height=h, color=ORANGE, label="Field score (2025, AI-calculated)", zorder=2)
    ax.barh([i - h / 2 - 0.02 for i in y], [r["rescored"] for r in rows], height=h, color=BLUE, label="Re-scored (sns-score-v1.1, CHIRPS)", zorder=2)
    ax.scatter([r["null_mean"] for r in rows], [i - h / 2 - 0.02 for i in y], marker="|", s=140, color=INK, lw=2,
               label="Zero-skill score (climatology)", zorder=3)
    ax.set_yticks(list(y))
    ax.set_yticklabels([f"{r['sign_id']}. {r['sign']}" + (" *" if r["status"] != "confirmed" else "") for r in rows])
    ax.set_xlim(0, 10); ax.set_ylim(-0.7, n + 0.2)
    ax.set_xlabel("Score out of 10")
    ax.grid(axis="x", color=GRID, lw=0.6, zorder=1)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False, fontsize=7.5)
    ttl = "Field scores vs re-scored, with the score a no-skill sign would get"
    ax.set_title(ttl + (f"  [{label}]" if label else ""), loc="left", fontsize=10, color=INK, pad=24)
    fig.text(0.01, -0.02, "* timing provisional (see PREREGISTRATION.md)", color=INK2, fontsize=7)
    _save(fig, out, "fig_scores")


def fig_windows(out, primary_rows, days, label, first_year=1981):
    rows = [r for r in primary_rows if not r.get("error")]
    if not rows:
        return
    cols = 3
    nr = (len(rows) + cols - 1) // cols
    fig, axes = plt.subplots(nr, cols, figsize=(7.2, 2.1 * nr), squeeze=False)
    for ax, r in zip(axes.flat, rows):
        ty = int(r["start"][:4])
        stat, xl = {"heavy_rain": (lambda w: w.maxDayMm, "wettest day (mm)"),
                    "rain_onset": (lambda w: w.max3DayMm, "wettest 3 days (mm)")}.get(
                        r["outcome"], (lambda w: w.totalMm, "window total (mm)"))
        vals = []
        for y in range(first_year, ty):
            s = stats_for(days, shift_years(r["start"], y - ty), shift_years(r["end"], y - ty), y)
            if s.complete:
                vals.append(stat(s))
        t = stats_for(days, r["start"], r["end"], ty)
        ax.hist(vals, bins=12, color=SEQ[1], edgecolor=SURF, lw=1)
        ax.axvline(stat(t), color=BLUE if r["happened"] else ORANGE, lw=2)
        ax.set_title(f"{r['sign_id']}. {r['sign'][:30]}\n{r['outcome']}, {r['start']} to {r['end'][5:]}", fontsize=7.5, loc="left")
        ax.tick_params(labelsize=7)
        ax.set_xlabel(xl, fontsize=7)
    for ax in list(axes.flat)[len(rows):]:
        ax.axis("off")
    fig.suptitle("Target window (line) against the same window in every earlier year since 1981 (bars)" + (f"  [{label}]" if label else ""),
                 x=0.01, ha="left", fontsize=10)
    from matplotlib.lines import Line2D
    fig.legend([Line2D([], [], color=BLUE, lw=2), Line2D([], [], color=ORANGE, lw=2)],
               ["prediction came true", "prediction did not come true"], loc="lower center", ncol=2, frameon=False,
               fontsize=7.5, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    _save(fig, out, "fig_windows")


def fig_sensitivity(out, sens_rows, sign_rows, label):
    ok = [r for r in sens_rows if r.get("error") == "" and r["variant"] == "pre-registered test"]
    if not ok:
        return
    configs = sorted({r["config"] for r in ok})
    signs = [r["sign_id"] for r in sorted(sign_rows, key=lambda r: r["sign_id"])]
    names = {r["sign_id"]: r["sign"] for r in sign_rows}
    M = []
    for s in signs:
        row = []
        for c in configs:
            v = [r["score"] for r in ok if r["sign_id"] == s and r["config"] == c]
            row.append(st.mean(v) if v else float("nan"))
        M.append(row)
    cmap = LinearSegmentedColormap.from_list("seq", SEQ)
    fig, ax = plt.subplots(figsize=(1.0 + 0.62 * len(configs), 0.3 * len(signs) + 1.6))
    im = ax.imshow(M, cmap=cmap, vmin=0, vmax=10, aspect="auto")
    for i, row in enumerate(M):
        for j, v in enumerate(row):
            if v == v:
                ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=6.5, color=INK if v < 6 else "#ffffff")
    ax.set_xticks(range(len(configs)))
    ax.set_xticklabels([c.replace("/", "\n") for c in configs], fontsize=6.5)
    ax.set_yticks(range(len(signs)))
    ax.set_yticklabels([f"{s}. {names[s][:28]}" for s in signs], fontsize=7)
    for sp in ax.spines.values():
        sp.set_visible(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cb.set_label("score", fontsize=7); cb.ax.tick_params(labelsize=6.5)
    ax.set_title("Score by dataset, location and reference period" + (f"  [{label}]" if label else ""), loc="left", fontsize=10)
    _save(fig, out, "fig_sensitivity")


def make_all(out, sign_rows, primary_rows, null_rows, sens_rows, days, spec, label=""):
    if sign_rows:
        fig_scores(out, sign_rows, label)
    fig_windows(out, primary_rows, days, label, spec["null_model"]["first_year"])
    if sign_rows:
        fig_sensitivity(out, sens_rows, sign_rows, label)

### Extraction module: rainfall APIs  (`fetch_data.py`)

In [ ]:
%%writefile fetch_data.py
"""Download the daily rainfall series used by rescore.py.

Every series is saved as data/<dataset>__<location>.csv (columns: date, mm) and logged in
data/manifest.json with the exact request, the download time and a SHA-256 of the file,
so the paper's data statement can cite exactly what was used.

Sources used in the paper (directly from the data providers' catalogues):
  --source gee           CHIRPS v2.0 (UCSB-CHG/CHIRPS/DAILY), CHIRPS v3.0 (UCSB-CHC/CHIRPS/V3/DAILY_SAT) and
                         ERA5-Land (ECMWF/ERA5_LAND/DAILY_AGGR) from the Google Earth Engine data catalog
                         (free for research; needs an Earth Engine-enabled Google Cloud project)
  --source cds           ERA5 hourly precipitation from the Copernicus Climate Data Store
                         (reanalysis-era5-single-levels; free CDS account and ~/.cdsapirc)
Mirrors (no account; for quick checks only, not used for the paper's numbers):
  --source climateserv   CHIRPS v2.0 through NASA/USAID SERVIR ClimateSERV
  --source openmeteo     ERA5 and ERA5-Land through the Open-Meteo Historical Weather API

Example:
  python fetch_data.py --source gee --gee-project my-ee-project --source cds
"""
from __future__ import annotations

import argparse
import datetime as dt
import hashlib
import json
import os
import sys
import time

import yaml

HERE = os.path.dirname(os.path.abspath(__file__))
DATA = os.path.join(HERE, "data")
START = "1981-01-01"
END = "2026-06-30"   # covers every target window in specs.yaml (latest ends 2026-05-31)


def load_locations():
    spec = yaml.safe_load(open(os.path.join(HERE, "specs.yaml"), encoding="utf-8"))
    return spec["locations"]


def write_series(dataset, location, rows, request_info):
    os.makedirs(DATA, exist_ok=True)
    rows = sorted(rows)
    path = os.path.join(DATA, f"{dataset}__{location}.csv")
    with open(path, "w", encoding="utf-8") as f:
        f.write("date,mm\n")
        for d, v in rows:
            f.write(f"{d},{'' if v is None else round(float(v), 3)}\n")
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()
    man_path = os.path.join(DATA, "manifest.json")
    man = json.load(open(man_path)) if os.path.exists(man_path) else {}
    missing = sum(1 for _, v in rows if v is None)
    man[f"{dataset}__{location}"] = dict(
        file=os.path.basename(path), sha256=sha, rows=len(rows), missing=missing,
        first=rows[0][0] if rows else None, last=rows[-1][0] if rows else None,
        downloaded_utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), request=request_info)
    json.dump(man, open(man_path, "w"), indent=2)
    print(f"  saved {path}: {len(rows)} days, {missing} missing, {rows[0][0]} .. {rows[-1][0]}")


def box_points(b, step=0.25):
    """Grid points every 0.25 deg (the ERA5 grid) inside the box."""
    import math
    lats, lons = [], []
    la = math.ceil(b["lat_min"] / step) * step
    while la <= b["lat_max"] + 1e-9:
        lats.append(round(la, 4)); la += step
    lo = math.ceil(b["lon_min"] / step) * step
    while lo <= b["lon_max"] + 1e-9:
        lons.append(round(lo, 4)); lo += step
    return [(a, o) for a in lats for o in lons]


def _get(session, url, params, tries=5):
    for i in range(tries):
        try:
            r = session.get(url, params=params, timeout=120)
            if r.status_code == 429:
                time.sleep(30 * (i + 1)); continue
            r.raise_for_status()
            return r
        except Exception as e:  # network hiccup: back off and retry
            if i == tries - 1:
                raise
            print("   retrying after error:", e)
            time.sleep(5 * (i + 1))


# ---------------------------------------------------------------- Open-Meteo (ERA5, ERA5-Land)
def fetch_openmeteo(locs):
    import requests
    s = requests.Session()
    url = "https://archive-api.open-meteo.com/v1/archive"
    for model, dataset in [("era5", "era5"), ("era5_land", "era5_land")]:
        for name, loc in locs.items():
            pts = [(loc["lat"], loc["lon"])] if "lat" in loc else box_points(loc)
            print(f"Open-Meteo {model} {name}: {len(pts)} grid point(s)")
            sums, counts = {}, {}
            # one request per decade keeps responses small
            y0 = int(START[:4])
            while y0 <= int(END[:4]):
                a = f"{y0}-01-01"
                b = min(f"{y0 + 9}-12-31", END)
                params = dict(latitude=",".join(str(p[0]) for p in pts), longitude=",".join(str(p[1]) for p in pts),
                              start_date=a, end_date=b, daily="precipitation_sum", timezone="GMT", models=model)
                js = _get(s, url, params).json()
                if isinstance(js, dict):
                    js = [js]
                for item in js:
                    for d, v in zip(item["daily"]["time"], item["daily"]["precipitation_sum"]):
                        if v is not None:
                            sums[d] = sums.get(d, 0.0) + v
                            counts[d] = counts.get(d, 0) + 1
                        else:
                            sums.setdefault(d, 0.0); counts.setdefault(d, 0)
                y0 += 10
                time.sleep(1)
            rows = [(d, (sums[d] / counts[d]) if counts[d] == len(pts) else None) for d in sums]
            write_series(dataset, name, rows, dict(api=url, model=model, points=pts, daily="precipitation_sum",
                                                   timezone="GMT", aggregation="mean of points" if len(pts) > 1 else "point",
                                                   start=START, end=END))


# ---------------------------------------------------------------- ClimateSERV (CHIRPS v2.0)
CHIRPS_RES = 0.05


def chirps_pixel_center(lat, lon):
    import math
    clon = -180 + (math.floor((lon + 180) / CHIRPS_RES) + 0.5) * CHIRPS_RES
    clat = -50 + (math.floor((lat + 50) / CHIRPS_RES) + 0.5) * CHIRPS_RES
    return round(clat, 4), round(clon, 4)


def climateserv_polygon(loc):
    if "lat" in loc:  # a small square inside the single CHIRPS pixel that contains the point
        clat, clon = chirps_pixel_center(loc["lat"], loc["lon"])
        e = 0.02
        a, b, c, d = clon - e, clon + e, clat - e, clat + e
    else:
        a, b, c, d = loc["lon_min"], loc["lon_max"], loc["lat_min"], loc["lat_max"]
    return [[a, d], [b, d], [b, c], [a, c], [a, d]]


def fetch_climateserv(locs):
    import requests
    s = requests.Session()
    base = "https://climateserv.servirglobal.net/api/"
    for name, loc in locs.items():
        poly = climateserv_polygon(loc)
        geom = json.dumps({"type": "Polygon", "coordinates": [poly], "properties": {}}).replace(" ", "")
        print(f"ClimateSERV CHIRPS {name}: polygon {poly}")
        rows = {}
        for y in range(int(START[:4]), int(END[:4]) + 1):
            a = dt.date(y, 1, 1)
            b = min(dt.date(y, 12, 31), dt.date.fromisoformat(END))
            params = dict(datatype=0, begintime=a.strftime("%m/%d/%Y"), endtime=b.strftime("%m/%d/%Y"),
                          intervaltype=0, operationtype=5, dateType_Category="default",
                          isZip_CurrentDataType=False, geometry=geom)
            for attempt in range(4):
                try:
                    r = s.post(base + "submitDataRequest/", params=params, timeout=120)
                    r.raise_for_status()
                    job = json.loads(r.text)[0]
                    for _ in range(600):
                        prog = json.loads(_get(s, base + "getDataRequestProgress/", {"id": job}).text)[0]
                        if prog == 100 or prog == -1:
                            break
                        time.sleep(2)
                    if prog == -1:
                        raise RuntimeError("ClimateSERV reported a processing error")
                    js = json.loads(_get(s, base + "getDataFromRequest/", {"id": job}).text)
                    break
                except Exception as e:
                    if attempt == 3:
                        raise
                    print("   retrying", y, e); time.sleep(10)
            n = 0
            for g in js.get("data", []):
                d = g["date"]
                try:
                    iso = dt.datetime.strptime(d, "%m/%d/%Y").date().isoformat()
                except ValueError:
                    iso = d[:10]
                v = g["value"].get("avg") if isinstance(g["value"], dict) else g["value"]
                v = None if v is None or float(v) < -999 else float(v)
                rows[iso] = v; n += 1
            print(f"   {y}: {n} days")
        write_series("chirps", name, list(rows.items()), dict(api=base, datatype="0 (CHIRPS v2.0)", operation="Average",
                                                              polygon=poly, start=START, end=END))


# ---------------------------------------------------------------- Google Earth Engine (CHIRPS v2.0, v3.0)
GEE_COLLECTIONS = [
    # (dataset name, Earth Engine collection, band, factor to mm/day, scale in m)
    ("chirps", "UCSB-CHG/CHIRPS/DAILY", "precipitation", 1.0, 5566),            # CHIRPS v2.0 (Funk et al., 2015)
    ("chirps_v3", "UCSB-CHC/CHIRPS/V3/DAILY_SAT", "precipitation", 1.0, 5566),  # CHIRPS v3.0 (Funk et al., 2026)
    ("era5_land", "ECMWF/ERA5_LAND/DAILY_AGGR", "total_precipitation_sum", 1000.0, 11132),  # metres -> mm
]


def fetch_gee(locs, project, only=None):
    """CHIRPS v2.0, CHIRPS v3.0 and ERA5-Land daily totals, directly from the Google Earth Engine catalog."""
    import ee
    try:
        ee.Initialize(project=project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=project)
    for dataset, cid, band, factor, scale in GEE_COLLECTIONS:
        if only and dataset not in only:
            continue
        for name, loc in locs.items():
            geom = (ee.Geometry.Point([loc["lon"], loc["lat"]]) if "lat" in loc else
                    ee.Geometry.Rectangle([loc["lon_min"], loc["lat_min"], loc["lon_max"], loc["lat_max"]]))
            print(f"Earth Engine {cid} {name}")
            rows = {}
            for y in range(int(START[:4]), int(END[:4]) + 1):
                end = min(dt.date(y + 1, 1, 1), dt.date.fromisoformat(END) + dt.timedelta(days=1)).isoformat()
                col = ee.ImageCollection(cid).filterDate(f"{y}-01-01", end).select(band)
                if "lat" in loc:
                    fc = col.map(lambda im: ee.Feature(None, {
                        "d": im.date().format("YYYY-MM-dd"),
                        "v": im.sample(geom, scale).first().get(band)}))
                else:
                    fc = col.map(lambda im: ee.Feature(None, {
                        "d": im.date().format("YYYY-MM-dd"),
                        "v": im.reduceRegion(ee.Reducer.mean(), geom, scale).get(band)}))
                info = fc.getInfo()["features"]
                for f in info:
                    v = f["properties"].get("v")
                    rows[f["properties"]["d"]] = None if v is None else float(v) * factor
                print(f"   {y}: {len(info)} days")
            write_series(dataset, name, list(rows.items()), dict(
                api="Google Earth Engine", catalog=f"https://developers.google.com/earth-engine/datasets/catalog/{cid.replace('/', '_')}",
                collection=cid, band=band, to_mm_factor=factor, scale_m=scale, geometry=loc, start=START, end=END))


# ---------------------------------------------------------------- Copernicus Climate Data Store (ERA5)
def era5_daily_from_hourly(times, values_m):
    """ERA5 total precipitation is accumulated over the hour ending at each valid time (metres).
    The total for UTC day D is the sum of the values valid at D 01:00 ... D+1 00:00 (mm). Days without all
    24 hours are returned as None."""
    acc, cnt = {}, {}
    for t, v in zip(times, values_m):
        day = (t - dt.timedelta(hours=1)).date().isoformat()   # hour ending at t belongs to the day of t - 1 h
        if v is None:
            continue
        acc[day] = acc.get(day, 0.0) + float(v) * 1000.0
        cnt[day] = cnt.get(day, 0) + 1
    return {d: (acc[d] if cnt[d] == 24 else None) for d in acc}


def fetch_cds(locs):
    """ERA5 hourly total precipitation directly from the Copernicus Climate Data Store
    (dataset reanalysis-era5-single-levels, doi:10.24381/cds.adbb2d47), summed to UTC days.
    Needs a free CDS account and ~/.cdsapirc (see https://cds.climate.copernicus.eu/how-to-api)."""
    import cdsapi
    import numpy as np
    import xarray as xr
    c = cdsapi.Client()
    tmp = os.path.join(DATA, "cds_tmp"); os.makedirs(tmp, exist_ok=True)
    for name, loc in locs.items():
        if "lat" in loc:
            area = [loc["lat"] + 0.25, loc["lon"] - 0.25, loc["lat"] - 0.25, loc["lon"] + 0.25]
        else:
            area = [loc["lat_max"], loc["lon_min"], loc["lat_min"], loc["lon_max"]]
        print(f"Copernicus CDS ERA5 {name}: area {area}")
        times, vals = [], []
        for y in range(int(START[:4]), int(END[:4]) + 2):
            months = [f"{m:02d}" for m in range(1, 13)] if y <= int(END[:4]) else ["01"]
            days = [f"{d:02d}" for d in range(1, 32)] if y <= int(END[:4]) else ["01"]
            hours = [f"{h:02d}:00" for h in range(24)] if y <= int(END[:4]) else ["00:00"]
            if y > int(END[:4]) and END[5:] != "12-31":
                break
            target = os.path.join(tmp, f"era5_{name}_{y}.nc")
            if not os.path.exists(target):
                c.retrieve("reanalysis-era5-single-levels", {
                    "product_type": ["reanalysis"], "variable": ["total_precipitation"],
                    "year": [str(y)], "month": months, "day": days, "time": hours,
                    "data_format": "netcdf", "download_format": "unarchived", "area": area}, target)
            ds = xr.open_dataset(target)
            tname = "valid_time" if "valid_time" in ds.coords else "time"
            da = ds["tp"]
            if "lat" in loc:
                da = da.sel(latitude=loc["lat"], longitude=loc["lon"], method="nearest")
            else:
                da = da.mean(dim=["latitude", "longitude"])
            for t, v in zip(ds[tname].values, da.values):
                times.append(dt.datetime.utcfromtimestamp((t - np.datetime64(0, "s")) / np.timedelta64(1, "s")))
                vals.append(None if np.isnan(v) else float(v))
            print(f"   {y}: {len(ds[tname])} hours")
        daily = era5_daily_from_hourly(times, vals)
        daily = {d: v for d, v in daily.items() if START <= d <= END}
        write_series("era5", name, list(daily.items()), dict(
            api="Copernicus Climate Data Store", dataset="reanalysis-era5-single-levels",
            doi="10.24381/cds.adbb2d47", variable="total_precipitation", area=area,
            aggregation="hourly accumulations summed to UTC days; " + ("nearest grid point" if "lat" in loc else "mean of grid points in box"),
            start=START, end=END))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--source", action="append", choices=["gee", "cds", "climateserv", "openmeteo"], required=True)
    ap.add_argument("--gee-project", help="Google Cloud project with Earth Engine enabled (for --source gee)")
    a = ap.parse_args()
    locs = load_locations()
    for src in a.source:
        if src == "openmeteo":
            fetch_openmeteo(locs)
        elif src == "climateserv":
            fetch_climateserv(locs)
        elif src == "cds":
            fetch_cds(locs)
        elif src == "gee":
            if not a.gee_project:
                sys.exit("--gee-project is required for --source gee")
            fetch_gee(locs, a.gee_project)
    print("Done. Next: python rescore.py")


if __name__ == "__main__":
    main()

### Requirements  (`requirements.txt`)

In [ ]:
%%writefile requirements.txt
pyyaml>=6
requests>=2.31
matplotlib>=3.8
# only for --source gee:
# earthengine-api>=1.0

### Self-test: verification measures  (`tests/test_standard.py`)

In [ ]:
%%writefile tests/test_standard.py
"""Checks the standard verification measures against hand-computed values and checks that the
exact Poisson-binomial test is calibrated when signs have no skill."""
import os, sys, random
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from standard_metrics import verify, poisson_binomial_sf
from math import comb
from math import isclose as _ic
isclose = lambda a, b, rel_tol=None: _ic(a, b, abs_tol=1.5e-3) if rel_tol is None else _ic(a, b, rel_tol=rel_tol)

# 1. Poisson-binomial with equal p equals the binomial
for n, p, c in [(10, 0.3, 4), (14, 0.5, 10), (5, 0.9, 5)]:
    exact = sum(comb(n, k) * p**k * (1-p)**(n-k) for k in range(c, n+1))
    assert isclose(poisson_binomial_sf([p]*n, c), exact, rel_tol=1e-12)
# 2. Hand example: 4 tests, outcomes 1,1,0,1, base rates .9,.5,.2,.33
v = verify([1,1,0,1], [0.9,0.5,0.2,0.33])
pc, e = 0.75, (0.9+0.5+0.2+0.33)/4
assert isclose(v["PC"], round(pc,3)) and isclose(v["PC_chance"], round(e,3))
assert isclose(v["HSS"], round((pc-e)/(1-e),3))
bs = 1/4; bsc = ((0.9-1)**2+(0.5-1)**2+(0.2-0)**2+(0.33-1)**2)/4
assert isclose(v["BSS"], round(1-bs/bsc,3))
print("hand-computed checks: PASS")

# 3. Calibration: hits drawn with exactly the climatological probabilities (no skill)
rng = random.Random(1)
rej05 = rej10 = 0; T = 20000
for _ in range(T):
    b = [rng.choice([0.2, 0.33, 0.5, 0.67, 0.9, 0.97]) for _ in range(14)]
    o = [1 if rng.random() < x else 0 for x in b]
    p = verify(o, b)["p_exact"]
    rej05 += p <= 0.05; rej10 += p <= 0.10
print(f"no-skill sets: p<=0.05 in {rej05/T:.3f}, p<=0.10 in {rej10/T:.3f}")
assert rej05/T <= 0.05 + 0.005 and rej10/T <= 0.10 + 0.007
print("calibration: PASS (exact test is not over-optimistic)")

### Self-test: significance calibration  (`tests/test_calibration.py`)

In [ ]:
%%writefile tests/test_calibration.py
"""The p-values must be honest: when rainfall has nothing to do with the sign, a test at the 5% level
should flag 'skill' in about 5% of cases or fewer. Runs each pre-registered primary test on 200
independent synthetic climates and reports the false-positive rate."""
import os, random, sys, datetime as dt
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import yaml
from rescore import run_test

PW = [0.10,0.15,0.35,0.55,0.50,0.55,0.75,0.75,0.65,0.35,0.15,0.08]; MU = [4,5,7,9,8,8,10,10,8,7,5,4]
def climate(seed):
    rng = random.Random(seed); d = dt.date(1981,1,1); m = {}; f = {}
    while d <= dt.date(2026,6,30):
        k = (d.year, d.month in (3,4,5)); f.setdefault(k, rng.lognormvariate(0,0.25))
        mo = d.month-1
        m[d.isoformat()] = round(rng.expovariate(1/(MU[mo]*f[k])),1) if rng.random() < min(0.95, PW[mo]*f[k]) else 0.0
        d += dt.timedelta(days=1)
    return m

spec = yaml.safe_load(open(os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "specs.yaml")))
tests = [t for s in spec["signs"] if s["status"] != "direction_needed" for t in s["tests"]]
n, hits05, hits10, total = 200, 0, 0, 0
for seed in range(n):
    days = climate(seed)
    for t in tests:
        _, _, _, _, p = run_test(days, t, 30, 1981)
        total += 1; hits05 += p <= 0.05; hits10 += p <= 0.10
print(f"{total} null tests: p<=0.05 in {hits05/total:.3f}, p<=0.10 in {hits10/total:.3f}")
assert hits05/total <= 0.07 and hits10/total <= 0.13, "p-values are anti-conservative"
print("PASS: p-values are calibrated (not over-optimistic)")

## Step 2. Install and self-test
The self-tests check the verification measures against hand-computed values and confirm that the significance test gives false alarms at the expected 5% rate on simulated no-skill data.

In [ ]:
!pip -q install pyyaml requests matplotlib pandas earthengine-api cdsapi xarray netCDF4
!python tests/test_standard.py
!python tests/test_calibration.py

## Step 3. Settings

In [ ]:
EE_PROJECT = ''   # REQUIRED: your Earth Engine-enabled Google Cloud project ID, e.g. 'sign-and-sky-ee'
CDS_KEY    = ''   # OPTIONAL: your Copernicus CDS API key (profile page), for the ERA5 check
assert EE_PROJECT, 'Please enter your Earth Engine project ID in EE_PROJECT'

## Step 4. Extract: CHIRPS v2.0, CHIRPS v3.0 and ERA5-Land from Google Earth Engine
A sign-in window opens once. Every series is saved to `data/` with its request, time and SHA-256 checksum in `data/manifest.json`.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
!python fetch_data.py --source gee --gee-project {EE_PROJECT}

## Step 5. Extract: ERA5 from the Copernicus Climate Data Store (optional)
Skipped automatically if `CDS_KEY` is empty. This takes the longest step (about 20–40 minutes in the CDS queue).

In [ ]:
if CDS_KEY:
    open('/root/.cdsapirc', 'w').write('url: https://cds.climate.copernicus.eu/api\nkey: ' + CDS_KEY + '\n')
    !python fetch_data.py --source cds
else:
    print('No CDS key: ERA5 sensitivity check skipped (CHIRPS and ERA5-Land are enough for the main result).')

## Step 6. Test and validate all 17 signs, exactly as pre-registered

In [ ]:
!python rescore.py
from IPython.display import Markdown, display
display(Markdown(open('results/summary.md', encoding='utf-8').read()))

## Step 7. Result for each of the 17 signs
One row per pre-registered test, on the primary dataset (CHIRPS v2.0, Offa Woreda centre pixel, 30 reference years).

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 40)
p = pd.read_csv('results/primary.csv')
cols = ['sign_id', 'sign', 'status', 'outcome', 'start', 'end', 'total_mm',
        'happened', 'percentile', 'score', 'trust', 'base_rate', 'null_mean', 'p_value', 'error']
display(p[[c for c in cols if c in p.columns]])
print('Tests run:', p['score'].notna().sum(), ' | came true:', int(p['happened'].fillna(0).sum()))

### Signs 3, 4 and 8: tested under every possible reading
Farmers' reading of these three signs is still being confirmed (checklist item P4), so rule R4 tests every possible reading. None of them enters the main result until the reading is fixed.

In [ ]:
b = pd.read_csv('results/branches.csv')
display(b[['sign_id', 'sign', 'branch', 'obs', 'outcome', 'start', 'end', 'total_mm', 'score', 'null_mean', 'p_value']])

## Step 8. Validation against chance: standard verification measures

In [ ]:
display(pd.read_csv('results/standard.csv'))

## Step 9. Figures

In [ ]:
from IPython.display import Image
for f in ['fig_scores', 'fig_windows', 'fig_sensitivity']:
    if os.path.exists(f'results/{f}.png'):
        display(Image(f'results/{f}.png', width=900))

## Step 10. Download the results (send this file back)

In [ ]:
!rm -rf data/cds_tmp && cp -r data results/data && cp specs.yaml results/ && zip -qr /content/results.zip results
from google.colab import files
files.download('/content/results.zip')